# 03. TCO, break-even과 도입 적합성

목표: inference 단가만 비교하지 않고 초기 구축비, 운영비와 품질을 함께 평가합니다. 기사 수치와 명시적인 toy assumption을 구분합니다.

In [ ]:
def annual_tco(
    decisions_per_day: int,
    inference_cost_per_1k: float,
    annual_fixed_cost: float = 0.0,
    one_time_cost: float = 0.0,
    amortization_years: int = 1,
) -> float:
    variable = decisions_per_day * 365 / 1_000 * inference_cost_per_1k
    return variable + annual_fixed_cost + one_time_cost / amortization_years

# 기사값: inference 단가와 약 $500의 GPU training rental
# toy assumption: 데이터·개발·평가 초기비 $150k, 연 운영비 $100k
frontier_cost = 34.0
specialist_cost = 0.5
specialist_one_time = 500 + 150_000
specialist_annual_fixed = 100_000

for volume in (10_000, 100_000, 1_000_000, 10_000_000):
    frontier = annual_tco(volume, frontier_cost)
    specialist = annual_tco(
        volume,
        specialist_cost,
        annual_fixed_cost=specialist_annual_fixed,
        one_time_cost=specialist_one_time,
        amortization_years=2,
    )
    print(
        f"{volume:>10,d}/day | frontier=${frontier:>12,.0f} "
        f"specialist=${specialist:>12,.0f} saving=${frontier-specialist:>12,.0f}"
    )

In [ ]:
def break_even_decisions_per_day(
    frontier_per_1k: float,
    specialist_per_1k: float,
    specialist_extra_annual_cost: float,
) -> float:
    savings_per_decision = (frontier_per_1k - specialist_per_1k) / 1_000
    if savings_per_decision <= 0:
        return float("inf")
    return specialist_extra_annual_cost / savings_per_decision / 365

annualized_fixed = specialist_annual_fixed + specialist_one_time / 2
break_even = break_even_decisions_per_day(
    frontier_cost,
    specialist_cost,
    annualized_fixed,
)
print(f"toy assumption의 break-even: 약 {break_even:,.0f} decisions/day")
assert 14_000 < break_even < 15_000

In [ ]:
def candidate_score(*, volume, verifiability, stability, expert_agreement, data_rights, sensitivity):
    values = [volume, verifiability, stability, expert_agreement, data_rights, sensitivity]
    if any(not 0 <= value <= 1 for value in values):
        raise ValueError("모든 입력은 0과 1 사이여야 합니다.")
    weights = {
        "volume": 0.20,
        "verifiability": 0.25,
        "stability": 0.15,
        "expert_agreement": 0.15,
        "data_rights": 0.15,
        "sensitivity": 0.10,
    }
    return sum(weights[name] * value for name, value in locals().items() if name in weights)

workflows = {
    "catalog_review": dict(volume=1.0, verifiability=0.9, stability=0.8, expert_agreement=0.9, data_rights=0.9, sensitivity=0.7),
    "annual_strategy": dict(volume=0.1, verifiability=0.2, stability=0.4, expert_agreement=0.3, data_rights=0.9, sensitivity=0.8),
    "ticket_routing": dict(volume=0.9, verifiability=0.9, stability=0.8, expert_agreement=0.8, data_rights=1.0, sensitivity=0.5),
}

for name, inputs in workflows.items():
    print(f"{name:16s}: {candidate_score(**inputs):.3f}")

assert candidate_score(**workflows["catalog_review"]) > candidate_score(**workflows["annual_strategy"])

In [ ]:
configs = [
    {"name": "base_9b", "quality": 0.642, "cost": 0.50},
    {"name": "trained_9b", "quality": 0.873, "cost": 0.50},
    {"name": "frontier_cheap", "quality": 0.768, "cost": 19.0},
    {"name": "frontier_strong", "quality": 0.769, "cost": 34.0},
]

def is_dominated(candidate, others):
    return any(
        other["quality"] >= candidate["quality"]
        and other["cost"] <= candidate["cost"]
        and (other["quality"] > candidate["quality"] or other["cost"] < candidate["cost"])
        for other in others
        if other is not candidate
    )

pareto = [config["name"] for config in configs if not is_dominated(config, configs)]
print("기사 보고값 기준 Pareto frontier:", pareto)
assert pareto == ["trained_9b"]

## 결론

Break-even은 frontier 단가뿐 아니라 데이터·평가·운영 고정비에 매우 민감합니다. 실제 의사결정에서는 quality가 같은지, 실패 비용과 human review 비용이 포함됐는지, 라이선스와 데이터 권리가 명확한지 먼저 확인하세요.